# Debt waterfall & cash sweep

**Priority of payments**: cash pays senior cash interest first, then mandatory senior principal, then subordinated interest, then residual to equity. Stress with **lower UFCF**.

## Concept

We implement a stylized **payment cascade** with `min` / `max` DSL patterns (cash available, required coupons, principal). This is a teaching skeleton — live facilities need **day-counts**, **PIK toggles**, and **blocked / restricted** payment regimes.

In [ ]:
from finstack.statements import ModelBuilder, Evaluator, FinancialModelSpec
from finstack.statements_analytics import run_sensitivity, evaluate_scenario_set, goal_seek, trace_dependencies, explain_formula, run_variance, evaluate_dcf, run_corporate_analysis, pl_summary_report, credit_assessment_report
import json
print("Loaded finstack.statements + finstack.statements_analytics")

from finstack.statements import StatementResult
from finstack.statements_analytics import run_variance, evaluate_scenario_set

PERIODS = ["2025Q1", "2025Q2", "2025Q3", "2025Q4"]


def z(v):
    return list(zip(PERIODS, v))


b = ModelBuilder("waterfall-demo")
b.periods("2025Q1..Q4", "2025Q2")

b.value("cash_flow_available", z([20.0, 18.0, 16.0, 14.0]))
b.value("senior_balance", z([100.0, 95.0, 90.0, 85.0]))
b.value("sub_balance", z([40.0, 40.0, 38.0, 36.0]))
b.value("senior_rate", z([0.06, 0.06, 0.06, 0.06]))
b.value("sub_rate", z([0.10, 0.10, 0.10, 0.10]))
b.value("senior_sched_principal", z([5.0, 5.0, 5.0, 5.0]))

b.compute("senior_cash_interest", "senior_balance * senior_rate / 4")
b.compute("sub_cash_interest", "sub_balance * sub_rate / 4")

b.compute("cash_after_senior_int", "cash_flow_available - senior_cash_interest")
b.compute("senior_principal_paid", "min(max(cash_after_senior_int, 0), senior_sched_principal)")
b.compute("cash_after_senior", "cash_after_senior_int - senior_principal_paid")
b.compute("sub_int_paid", "min(max(cash_after_senior, 0), sub_cash_interest)")
b.compute("residual_to_equity", "cash_after_senior - sub_int_paid")

b.compute("senior_balance_eop", "senior_balance - senior_principal_paid")

spec = b.build()
base_res = Evaluator().evaluate(spec)
model_json = spec.to_json()
base_json = base_res.to_json()

print(base_res.to_polars_wide().select(
    [
        "period_id",
        "cash_flow_available",
        "senior_cash_interest",
        "senior_principal_paid",
        "sub_int_paid",
        "residual_to_equity",
        "senior_balance_eop",
    ]
))


In [ ]:
print("Waterfall stress + variance")
# Stress: lower cash flow via scenario set
scenarios = json.dumps(
    {
        "scenarios": {
            "base": {"overrides": {}},
            "stress": {"overrides": {"cash_flow_available": 9.0}},
        }
    }
)
mp = json.loads(evaluate_scenario_set(model_json, scenarios))

for name, ej in mp.items():
    r = StatementResult.from_json(json.dumps(ej))
    print("===", name, "===")
    for p in PERIODS:
        print(
            p,
            "residual_equity",
            r.get("residual_to_equity", p),
            "sub_int_paid",
            r.get("sub_int_paid", p),
        )

# Variance vs stressed re-eval
b2 = ModelBuilder("waterfall-stress")
b2.periods("2025Q1..Q4", "2025Q2")
b2.value("cash_flow_available", z([9.0, 9.0, 9.0, 9.0]))
b2.value("senior_balance", z([100.0, 95.0, 90.0, 85.0]))
b2.value("sub_balance", z([40.0, 40.0, 38.0, 36.0]))
b2.value("senior_rate", z([0.06, 0.06, 0.06, 0.06]))
b2.value("sub_rate", z([0.10, 0.10, 0.10, 0.10]))
b2.value("senior_sched_principal", z([5.0, 5.0, 5.0, 5.0]))
b2.compute("senior_cash_interest", "senior_balance * senior_rate / 4")
b2.compute("sub_cash_interest", "sub_balance * sub_rate / 4")
b2.compute("cash_after_senior_int", "cash_flow_available - senior_cash_interest")
b2.compute("senior_principal_paid", "min(max(cash_after_senior_int, 0), senior_sched_principal)")
b2.compute("cash_after_senior", "cash_after_senior_int - senior_principal_paid")
b2.compute("sub_int_paid", "min(max(cash_after_senior, 0), sub_cash_interest)")
b2.compute("residual_to_equity", "cash_after_senior - sub_int_paid")
b2.compute("senior_balance_eop", "senior_balance - senior_principal_paid")
stress_res = Evaluator().evaluate(b2.build())
var_cfg = json.dumps(
    {
        "baseline_label": "base",
        "comparison_label": "stress_cf",
        "metrics": ["residual_to_equity", "sub_int_paid", "senior_principal_paid"],
        "periods": ["2025Q3"],
    }
)
print(run_variance(base_json, stress_res.to_json(), var_cfg))


## Takeaways

- **Waterfalls** are `min`/`max` chains on **cash available**; ordering encodes **legal priority**.
- Stress tests expose **sub debt** getting partially paid or **equity residual** going negative in thin quarters.
- `run_variance` quantifies **policy vs stress** for selected metrics and periods.